# 1. Dataset preparation

Use images and matching UTF-8 expected-text files. No boxes are needed. Keep prices complete, including decimals and currency; current labels preserve case, spaces and symbols literally. The included fictional data has single/two-line plates and mixed-size price text. It is not a benchmark.

Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
subprocess.run([sys.executable, str(ROOT / "tools/make_dummy_data.py")], check=True)
from waterec_ocr.data_io import scan_pairs
from waterec_ocr.data import audit_splits
train = scan_pairs(config["dataset_root"] / "train", config["normalization"])
val = scan_pairs(config["dataset_root"] / "val", config["normalization"])
print(json.dumps(audit_splits(train, val), indent=2))


In [ ]:
from PIL import Image
from IPython.display import display
for row in train[:3]:
    print(row["label"])
    display(Image.open(row["path"]))


For real data, create `dataset/train/*.png + *.txt` and a separate `dataset/val` (or configure another root). The optional annotator at localhost:8000 saves image-level TXT labels. Never put final validation/OOD images in the fitting pool. The trainer creates its own internal training holdout. No cell deletes your data or weights.